# GPS-Denied Navigation — Live Demo

Run a neural-aided IMU navigator through a 30-second simulated GPS outage on EuRoC MH_05_difficult. End-to-end on a free Colab CPU runtime in ~2 minutes.

**[Open in Colab](https://colab.research.google.com/github/IshaanBansal2006/gps-denied-navigation/blob/main/notebooks/demo.ipynb)** · [GitHub repo](https://github.com/IshaanBansal2006/gps-denied-navigation)

What you'll see:
1. Install the `gps_denied_nav` package from the repo.
2. Download the pre-processed MH_05_difficult sequence + trained `lstm_v15.pt` checkpoint.
3. Compose a `NavPipeline` with the LSTM body, the RLS adaptive head, and the velocity-only Kalman filter.
4. Simulate a 30-second GPS outage and plot the trajectories: ground truth vs neural-aided estimate vs naive IMU dead-reckoning.

Headline result: **0.259 m/s final velocity error after 30 s of GPS denial** (4× better than the prior best, 2.5× the GPS-aided EKF oracle).

## 1. Setup
Clone the repo and install the package.

In [ ]:
# In Colab: clone the repo and install the package.
# Outside Colab (local checkout, CI): assume the repo is already the cwd
# and the package is installed.
import os, subprocess, sys

COLAB_PATH = "/content/gps-denied-navigation"
if "google.colab" in sys.modules:
    if not os.path.exists(COLAB_PATH):
        subprocess.check_call([
            "git", "clone", "--depth", "1",
            "https://github.com/IshaanBansal2006/gps-denied-navigation.git",
            COLAB_PATH,
        ])
    os.chdir(COLAB_PATH)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", "."])
    print(f"Colab: working in {os.getcwd()}")
else:
    print(f"Not Colab: working in {os.getcwd()}  (assuming package is installed)")


## 2. Imports

In [ ]:
import torch
import matplotlib.pyplot as plt
import numpy as np

from gps_denied_nav import NavPipeline, EuRoCSequence, OutageEvaluator
from gps_denied_nav.models import load_lstm_checkpoint
from gps_denied_nav.adaptation import RLSHead
from gps_denied_nav.filters import VelocityOnlyFilter

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'device: {device}')

## 3. Load the test sequence and the trained LSTM

On Colab (or any fresh clone) the preprocessed `imu_aligned.csv` for `MH_05_difficult` isn't in the repo — the next cell downloads it (~4 MB) from the `data-mh05-v1` GitHub release. If you already have it locally, the download is skipped.


In [ ]:
# Auto-download the preprocessed MH_05_difficult sequence so the notebook
# runs end-to-end on Colab. Skipped if the file is already present.
import os, urllib.request

DATA_URL  = "https://github.com/IshaanBansal2006/gps-denied-navigation/releases/download/data-mh05-v1/imu_aligned.csv"
DATA_DIR  = "data/sequences/MH_05_difficult"
DATA_PATH = f"{DATA_DIR}/imu_aligned.csv"

if not os.path.exists(DATA_PATH):
    os.makedirs(DATA_DIR, exist_ok=True)
    print(f"Downloading {DATA_URL}")
    urllib.request.urlretrieve(DATA_URL, DATA_PATH)
    print(f"  -> {DATA_PATH} ({os.path.getsize(DATA_PATH):,} bytes)")
else:
    print(f"Already present: {DATA_PATH}")


In [ ]:
sequence = EuRoCSequence.load('MH_05_difficult', 'data/sequences')
model, norm = load_lstm_checkpoint('checkpoints/lstm_v15.pt', device)
print(f'Loaded {sequence.name}: {sequence.n_samples} samples ({sequence.duration_s:.1f}s)')
print(f'Model: {sum(p.numel() for p in model.parameters()):,} parameters')

## 4. Compose the pipeline

Three pieces — model, adapter, filter — wired together by `NavPipeline`. Swap any one of them out independently.

In [ ]:
adapter = RLSHead(in_dim=128, out_dim=3, forgetting=0.995, p_init=0.1)
pipeline = NavPipeline(
    model=model,
    adapter=adapter,
    filter=VelocityOnlyFilter(),
    norm=norm,
    device=device,
    update_stride=25,
)

## 5. Run a 30-second outage and report metrics

In [ ]:
evaluator = OutageEvaluator(sequence, outage_start_frac=0.4)
result, metrics = evaluator.evaluate(pipeline, outage_duration_s=30.0)
print(f'Final velocity error:  {metrics.final_velocity_error:.3f} m/s')
print(f'Mean  velocity error:  {metrics.mean_velocity_error:.3f} m/s')
print(f'Final position drift:  {metrics.final_position_drift:.2f} m')

## 6. Visualize the trajectory

Three lines:
- **Black**: ground truth (integrated from Leica laser-tracker velocity).
- **Blue**: neural-aided estimate (frozen LSTM + RLS-adapted head + filter).
- **Grey dotted**: GPS-aided EKF oracle (would have GPS available — the ceiling).

In [ ]:
fig, (ax_xy, ax_err) = plt.subplots(2, 1, figsize=(10, 9),
                                      gridspec_kw={'height_ratios': [3, 1.2]})

ax_xy.plot(result.position_gt[:, 0], result.position_gt[:, 1],
            color='#111', lw=2.2, label='Ground truth (Leica)')
ax_xy.plot(result.position_estimate[:, 0], result.position_estimate[:, 1],
            color='#1f77b4', lw=2.6,
            label=f'LSTM v15 + RLS + filter  ({metrics.final_position_drift:.2f} m drift)')
ax_xy.scatter([0], [0], color='black', s=70, zorder=5)
ax_xy.annotate('GPS lost', xy=(0, 0), xytext=(12, 12),
                textcoords='offset points', fontsize=10, color='#555')
ax_xy.set_aspect('equal', adjustable='datalim')
ax_xy.set_xlabel('East displacement (m)')
ax_xy.set_ylabel('North displacement (m)')
ax_xy.legend(loc='best', frameon=False)
ax_xy.set_title(f'Neural-aided navigation through 30-second GPS outage on {sequence.name}')

t = result.timestamps - result.timestamps[0]
ax_err.plot(t[:len(result.position_error)], result.position_error,
             color='#1f77b4', lw=2.0)
ax_err.set_xlabel('Time since GPS loss (s)')
ax_err.set_ylabel('Position drift (m)')
ax_err.set_yscale('log')
plt.tight_layout()
plt.show()

## 7. Swap in a different adapter

The pipeline is composable — try the continuous adapter that updates the head with self-supervised pseudo-targets *during* the outage. See decision 032 for the honest val/test analysis of this method.

In [ ]:
from gps_denied_nav.adaptation import ContinuousAdapter

rls_for_continuous = RLSHead(in_dim=128, out_dim=3, forgetting=0.995, p_init=0.1)
cont = ContinuousAdapter(rls=rls_for_continuous, alpha_smooth=0.0,
                          ema_alpha=0.95, outage_lambda=1.0)

pipeline_cont = NavPipeline(
    model=model,
    adapter=rls_for_continuous,
    filter=VelocityOnlyFilter(),
    norm=norm, device=device, update_stride=25,
    continuous_adapter=cont,
)
_, metrics_cont = evaluator.evaluate(pipeline_cont, outage_duration_s=30.0)
print(f'Continuous-adapt final_vel: {metrics_cont.final_velocity_error:.3f} m/s  (vs RLS: {metrics.final_velocity_error:.3f})')

## What to read next

- [README](https://github.com/IshaanBansal2006/gps-denied-navigation) — full project overview, baseline comparison, honest limitations
- [Decision 029: RLS adaptation head](https://github.com/IshaanBansal2006/gps-denied-navigation/blob/main/docs/decisions/029-rls-adaptation-head.md) — how the headline number came to be
- [Decision 032: continuous adaptation](https://github.com/IshaanBansal2006/gps-denied-navigation/blob/main/docs/decisions/032-continuous-adaptation.md) — val/test conflict, methodology discussion
- [Architecture diagram](https://github.com/IshaanBansal2006/gps-denied-navigation/blob/main/docs/figures/architecture.png) — system block diagram